In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

In [3]:
os.getcwd()

'/content'

In [4]:
os.chdir('/content/drive/MyDrive/Colab Notebooks')

In [5]:
os.listdir()

['Untitled2.ipynb',
 'Untitled',
 '주피터&마크다운 튜토리얼.ipynb',
 'OpenAI API를 활용한 Prompt Engineering 심화 실습.ipynb',
 'gradio 실습 SeSAC.ipynb',
 'MCP 실습 SeSAC.ipynb',
 'Untitled0.ipynb',
 'mnist_mlp.ipynb',
 'MNIST_MLP_Colab.ipynb',
 'MNIST_Dataset.pkl',
 'trainer.py',
 'gradcamviz.py',
 '__pycache__',
 'Untitled1.ipynb',
 'CIFAR10_Dataset.pkl',
 'CIFAR10_CNN_Colab.ipynb']

In [8]:
import torch
from torchvision import transforms

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [10]:
from trainer import Trainer, set_seed

In [11]:
set_seed()

## 훈련셋 준비 : 층화추출용

In [16]:
from torchvision.datasets import CIFAR10

In [18]:
train_cifar10 = CIFAR10(root='../data', train=True, download=True)

100%|██████████| 170M/170M [00:04<00:00, 35.3MB/s]


In [19]:
train_targets = torch.tensor(data=train_cifar10.targets, dtype=torch.long)

In [20]:
train_targets.numel()

50000

In [21]:
n_per_class = 1000

In [22]:
sampled_indices = []

In [23]:
for class_id in range(10):
  class_indices = torch.where(train_targets == class_id)[0]
  shuffled_indices = class_indices[torch.randperm(n=len(class_indices))]
  selected_indices = shuffled_indices[:n_per_class]
  sampled_indices.append(selected_indices)

In [25]:
sampled_indices = torch.cat(sampled_indices, dim=0)

In [26]:
sampled_indices = sampled_indices[torch.randperm(n=len(sampled_indices))]

In [27]:
imagenet_avg = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

In [28]:
transform_train = transforms.Compose(
    transforms=[
        transforms.Resize(size=(128, 128)),
        transforms.RandomCrop(size=128, padding=8),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_avg, imagenet_std)
    ]
)

In [29]:
transform_test = transforms.Compose(
    transforms=[
        transforms.Resize(size=(128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_avg, imagenet_std)
    ]
)

In [30]:
train_cifar10 = CIFAR10(root='../data', train=True, transform=transform_train)

In [31]:
test_cifar10 = CIFAR10(root='../data', train=False, transform=transform_test)

## Subset 생성

In [32]:
from torch.utils.data import Subset

In [33]:
train_subset = Subset(dataset=train_cifar10, indices=sampled_indices.tolist())

In [34]:
len(train_subset)

10000

In [37]:
train_subset_targets = torch.tensor(train_subset.dataset.targets)

In [38]:
torch.bincount(input=train_subset_targets[train_subset.indices])

tensor([1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000])

## 배치 데이터 생성

In [39]:
from torch.utils.data import DataLoader

In [40]:
bs = 256

In [41]:
train_loader = DataLoader(dataset=train_subset, batch_size=bs, shuffle=True)
test_loader = DataLoader(dataset=test_cifar10, batch_size=bs, shuffle=False)

## 전이학습 모델 생성 및 확인

In [43]:
from torchvision import models
import torch.nn as nn

In [44]:
weights = models.ResNet18_Weights.DEFAULT

In [45]:
model = models.resnet18(weights=weights)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 153MB/s]


In [46]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
model.layer4[-1].conv2
# Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)

Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)

In [ ]:
sum(p.numel() for p in model.parameters() if p.requires_grad)
# 11689512

11689512

## 가중치 고정 및 마지막 분류기 교체

In [ ]:
for p in model.parameters():
    p.requires_grad = False

In [ ]:
sum(p.numel() for p in model.parameters() if p.requires_grad)
# 0

0

In [52]:
model.fc.out_features

1000

In [53]:
model.fc

Linear(in_features=512, out_features=1000, bias=True)

In [54]:
model.fc = nn.Linear(in_features=model.fc.in_features, out_features=10, bias=True)

In [55]:
model = model.to(device)

In [57]:
# 손실함수 및 최적화 알고리즘 생성
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.fc.parameters(), lr=0.001)

In [58]:
# resnet 모델 클래스 생성 및 학습
trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=test_loader,
    flatten=False,
    device=device
)

[Trainer] Using device: cuda


In [59]:
history = trainer.fit(10)

[Epoch 01] Loss = 1.7461, Train_Acc = 0.6197, Test_Acc = 0.6086
[Epoch 02] Loss = 1.1357, Train_Acc = 0.6835, Test_Acc = 0.6689
[Epoch 03] Loss = 0.9678, Train_Acc = 0.6966, Test_Acc = 0.6738
[Epoch 04] Loss = 0.8993, Train_Acc = 0.7172, Test_Acc = 0.7007
[Epoch 05] Loss = 0.8604, Train_Acc = 0.7326, Test_Acc = 0.7123
[Epoch 06] Loss = 0.8342, Train_Acc = 0.7349, Test_Acc = 0.7122
[Epoch 07] Loss = 0.8137, Train_Acc = 0.7387, Test_Acc = 0.7161
[Epoch 08] Loss = 0.7935, Train_Acc = 0.7470, Test_Acc = 0.7179
[Epoch 09] Loss = 0.7747, Train_Acc = 0.7498, Test_Acc = 0.7274
[Epoch 10] Loss = 0.7597, Train_Acc = 0.7507, Test_Acc = 0.7243
Total training time: 6 min 28 sec


## 미세 조정

In [ ]:
for p in model.layer4.parameters():
    p.requires_grad = True  

In [61]:
for p in model.fc.parameters():
  p.requires_grad = True

In [66]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

8398858

In [62]:
params_fine_tuning = filter(lambda p: p.requires_grad, model.parameters())

In [63]:
optimizer = torch.optim.Adam(params=params_fine_tuning, lr=0.0001)

In [64]:
trainer.optimizer = optimizer

In [65]:
history = trainer.fit(5)

[Epoch 01] Loss = 0.6095, Train_Acc = 0.8593, Test_Acc = 0.8137
[Epoch 02] Loss = 0.3867, Train_Acc = 0.9038, Test_Acc = 0.8380
[Epoch 03] Loss = 0.3030, Train_Acc = 0.9205, Test_Acc = 0.8447
[Epoch 04] Loss = 0.2504, Train_Acc = 0.9369, Test_Acc = 0.8558
[Epoch 05] Loss = 0.2069, Train_Acc = 0.9502, Test_Acc = 0.8544
Total training time: 3 min 16 sec


## 모델 파라미터 저장

In [67]:
os.getcwd()

'/content/drive/MyDrive/Colab Notebooks'

In [68]:
os.chdir('../data')

In [69]:
model.state_dict()

OrderedDict([('conv1.weight',
              tensor([[[[-1.0419e-02, -6.1356e-03, -1.8098e-03,  ...,  5.6615e-02,
                          1.7083e-02, -1.2694e-02],
                        [ 1.1083e-02,  9.5276e-03, -1.0993e-01,  ..., -2.7124e-01,
                         -1.2907e-01,  3.7424e-03],
                        [-6.9434e-03,  5.9089e-02,  2.9548e-01,  ...,  5.1972e-01,
                          2.5632e-01,  6.3573e-02],
                        ...,
                        [-2.7535e-02,  1.6045e-02,  7.2595e-02,  ..., -3.3285e-01,
                         -4.2058e-01, -2.5781e-01],
                        [ 3.0613e-02,  4.0960e-02,  6.2850e-02,  ...,  4.1384e-01,
                          3.9359e-01,  1.6606e-01],
                        [-1.3736e-02, -3.6746e-03, -2.4084e-02,  ..., -1.5070e-01,
                         -8.2230e-02, -5.7828e-03]],
              
                       [[-1.1397e-02, -2.6619e-02, -3.4641e-02,  ...,  3.2521e-02,
                          6.6221

In [70]:
file_path = 'CIFAR10_ResNet18.pth'

In [71]:
torch.save(obj=model.state_dict(), f=file_path)

In [72]:
os.listdir()

['cifar-10-python.tar.gz', 'cifar-10-batches-py', 'CIFAR10_ResNet18.pth']